In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import catboost as cb
from processor import PolarsLoader, ExprProcessor, PandasConverter
from IPython.display import Markdown

In [3]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(3, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')
with open('grade_subgrade.pkl', 'rb') as f:
    c_map = pkl.load(f)
df_train['grade_subgrade_no'] = df_train['grade_subgrade'].map(c_map).astype('int')
df_test['grade_subgrade_no'] = df_test['grade_subgrade'].map(c_map).astype('int')
df_train.shape, df_test.shape

((593994, 13), (254569, 12))

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade_no']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
import importlib
from modeler import Experimenter

In [7]:
e = Experimenter(df_train.sample(frac = 0.01, random_state = 123), 'exp', sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [8]:
from modeler._metric import Metric
from modeler._stacking import Stacking
e.add_metric('AUC', [(None, y)], slice(-1, None), roc_auc_score, include_train = True)
e.add_metric('AUC2', [(None, y)], slice(None, 1), roc_auc_score, include_train = True)
e.add_stacking('S1',[(None, y)], slice(-1, None))
e.add_stacking('S2', [(None, y)], slice(None, 1))

In [9]:
Markdown(
    e.desc_spec()
)

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=3, random_state=123, shuffle=True)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=123)` |
| **Splitter Params** | `{y='loan_paid_back'}` |
| **Outer Folds** | 3 |
| **Inner Folds** | 1 |

In [10]:
e.set_grp('clf', 'exp', parent_grp = None, edges = [(None, [y])], y = y, method = 'predict_proba')
e.set_grp('preprocessor', 'pipe', parent_grp = None, method = 'transform')

In [11]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.set_node('ohe', 'preprocessor', OneHotEncoder, edges = [(None, X_cat)], params={'sparse_output': False})
e.build()

Building 2 node(s)
Build 3/3 (100%)> std 2/2 (100%)
Build complete: 2 node(s)


In [12]:
e.rename_grp('preprocessor', 'preproc')

In [13]:
e.rename_grp('preproc', 'preprocessor')

In [14]:
e.build()

Building 0 node(s)
Build 3/3 (100%)> Node 0
Build complete: 0 node(s)


In [15]:
e.build(rebuild=True)

Building 2 node(s)
Build 3/3 (100%)> std 2/2 (100%)
Build complete: 2 node(s)


In [16]:
from sklearn.linear_model import LogisticRegression

e.set_grp('lr', parent_grp = 'clf', processor = LogisticRegression)

In [17]:
from modeler import col
e.set_node('lr1', 'lr', edges = [('std', None)])
e.set_node('lr2', 'lr', edges = [('std', None), ('ohe', col.ohe_drop_first)])

In [18]:
e.build()

Building 0 node(s)
Build 3/3 (100%)> Node 0
Build complete: 0 node(s)


In [19]:
results = e.nodes['lr1'].experiment(0, ['output'])
results

<generator object Node.experiment at 0x7f563c179fe0>

In [20]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [21]:
Markdown(
    e.desc_node('lr2', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr2["lr2"]
        lr2_dummy[ ]
        style lr2_dummy fill:none,stroke:none
    end
    style node_lr2 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr2
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr2
    node_std --> node_lr2
```

**Path from Root to 'lr2' (3 path(s) found)**

In [22]:
e.set_grp('dim_reduction', parent_grp = 'preprocessor')

In [23]:
e.nodes['std'].output_edges

['lr1', 'lr2']

In [24]:
from sklearn.decomposition import PCA
e.set_node('pca', 'dim_reduction', processor=PCA, edges = [('std', None)], params={'n_components': 0.9})

In [25]:
e.nodes['std'].output_edges, e.nodes['pca'].grp.role, e.nodes['pca'].status

(['lr1', 'lr2', 'pca'], 'pipe', None)

In [26]:
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.build()

Effected 3 dependent node(s): ['lr1', 'lr2', 'pca']
Building 2 node(s)
Build 3/3 (100%)> pca 2/2 (100%)
Build complete: 2 node(s)


In [27]:
e.exp('lr*', retry=True)

Experimenting 2 node(s)
Exp 3/3 (100%)> lr2 2/2 (100%)
Experimentation complete: 2 node(s)


In [28]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [29]:
e.set_node('lr3', 'lr', edges = [('ohe', None), ('pca', None)])

In [30]:
Markdown(
    e.desc_node('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [31]:
Markdown(
    e.desc_node('lr3', direction = 'LR', show_params=True)
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr>"]
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>PCA</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>n_components</b></td><td align='left'>0.9</td></tr></table>"]
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [32]:
e.set_grp('cb', parent_grp='clf', processor=cb.CatBoostClassifier, params={'verbose': 0})

In [33]:
e.set_node('cb1',  grp = 'cb', edges = [(None, X_num), (None, X_cat)], params = {'cat_features': X_cat})

In [34]:
import lightgbm as lgb

In [35]:
e.set_grp('lgb', parent_grp='clf', processor=lgb.LGBMClassifier, params={'verbose': -1})

In [36]:
e.set_node('lgb1',  grp = 'lgb', edges = [(None, X_num), (None, X_cat)], params={'categorical_features': X_cat})

In [37]:
import xgboost as xgb
e.set_grp('xgb', parent_grp='clf', processor=xgb.XGBClassifier, params={'verbose': -1})

In [38]:
e.set_node('xgb1',  grp = 'xgb', edges = [(None, X_num), (None, X_cat)], params={'enable_categorical': True})

In [39]:
e.exp(None)

Experimenting 4 node(s)
Exp 3/3 (100%)> lr3 4/4 (100%) > 100/100 (100%) training-binary_logloss: 0.0738, valid_1-binary_logloss: 0.2509
Experimentation complete: 4 node(s)


In [40]:
e.exp(None)

Experimenting 0 node(s)
Exp 3/3 (100%)> Node 0
Experimentation complete: 0 node(s)


In [41]:
e.nodes['lr1'].status

'built'

In [42]:
e.stacking['S1'].get_dataset(None)

,lr4__loan_paid_back_1,lr3__loan_paid_back_1,xgb1__loan_paid_back_1,lgb1__loan_paid_back_1,lr1__loan_paid_back_1,lr2__loan_paid_back_1,cb1__loan_paid_back_1,loan_paid_back
id,,,,,,,,
176836,0.713658,0.712976,0.727636,0.679160,0.566541,0.711601,0.526876,1
321322,0.913602,0.949090,0.989303,0.969608,0.839653,0.910697,0.944185,0
225907,0.781523,0.744901,0.247498,0.396050,0.656371,0.779841,0.540851,0
290104,0.691744,0.689015,0.495386,0.661095,0.572986,0.709593,0.514444,1
453658,0.942474,0.932390,0.960040,0.951571,0.827431,0.940926,0.934432,1
...,...,...,...,...,...,...,...,...
425863,0.966873,0.961187,0.999113,0.993941,0.914222,0.964557,0.987203,1
103194,0.977952,0.960983,0.997141,0.996805,0.940460,0.976801,0.971427,1
252332,0.009931,0.008886,0.000640,0.005673,0.477469,0.011397,0.008119,0


In [43]:
e.stacking['S1']._get_nodes(None)

['lr4', 'lr3', 'xgb1', 'lgb1', 'lr1', 'lr2', 'cb1']

In [44]:
e.stacking['S2'].get_dataset(None)

,lr4__loan_paid_back_0,lr3__loan_paid_back_0,xgb1__loan_paid_back_0,lgb1__loan_paid_back_0,lr1__loan_paid_back_0,lr2__loan_paid_back_0,cb1__loan_paid_back_0,loan_paid_back
id,,,,,,,,
176836,0.286342,0.287024,0.272364,0.320840,0.433459,0.288399,0.473124,1
321322,0.086398,0.050910,0.010697,0.030392,0.160347,0.089303,0.055815,0
225907,0.218477,0.255099,0.752502,0.603950,0.343629,0.220159,0.459149,0
290104,0.308256,0.310985,0.504614,0.338905,0.427014,0.290407,0.485556,1
453658,0.057526,0.067610,0.039960,0.048429,0.172569,0.059074,0.065568,1
...,...,...,...,...,...,...,...,...
425863,0.033127,0.038813,0.000887,0.006059,0.085778,0.035443,0.012797,1
103194,0.022048,0.039017,0.002859,0.003195,0.059540,0.023199,0.028573,1
252332,0.990069,0.991114,0.999360,0.994327,0.522531,0.988603,0.991881,0


In [45]:
e.metric['AUC2'].get_metrics(None)

0                             1                             2  \
             0                             0                             0   
         valid train_sub valid_sub     valid train_sub valid_sub     valid   
lr1   0.222494  0.218787  0.235036  0.212007  0.228005  0.197061  0.231907   
lr2   0.076601  0.082250  0.094238  0.086865  0.081280  0.056104  0.087830   
cb1   0.074635  0.058834  0.088049  0.085150  0.061278  0.061374  0.092149   
xgb1  0.093837  0.000000  0.137643  0.110998  0.000000  0.110250  0.115384   
lgb1  0.084511  0.000489  0.116080  0.101826  0.000114  0.093280  0.105895   
lr3   0.077149  0.084252  0.096674  0.087862  0.083204  0.056423  0.093097   

                          
                          
     train_sub valid_sub  
lr1   0.214295  0.216667  
lr2   0.076200  0.098750  
cb1   0.033871  0.085733  
xgb1  0.000004  0.096035  
lgb1  0.000415  0.082498  
lr3   0.077231  0.094198

In [46]:
e.nodes['lr1'].status

'built'

In [47]:
e.finalize('lr1')

Finalize 'lr1'


In [48]:
e.nodes['lr1'].status

'finalized'

In [49]:
del e

In [50]:
e = Experimenter.load('exp', df_train.sample(frac = 0.01, random_state = 123))

Loaded: 9 node(s), 7 group(s), 3 fold(s)


In [51]:
e.metric

{'AUC': <modeler._metric.Metric at 0x7f55b1b55d00>,
 'AUC2': <modeler._metric.Metric at 0x7f55b0225280>}

In [52]:
e.stacking['S1']._get_nodes(None)

['lr4', 'lr3', 'xgb1', 'lgb1', 'lr1', 'lr2', 'cb1']

In [53]:
e.nodes['lr1'].status

'finalized'

In [54]:
df_input, df_output = e.desc_node_vars('lr3', 0)
display(df_input)
display(df_output)

name
node seq                                      
ohe  0                      ohe__gender_Female
     1                        ohe__gender_Male
     2                       ohe__gender_Other
     3            ohe__marital_status_Divorced
     4             ohe__marital_status_Married
     5              ohe__marital_status_Single
     6             ohe__marital_status_Widowed
     7         ohe__education_level_Bachelor's
     8        ohe__education_level_High School
     9           ohe__education_level_Master's
     10             ohe__education_level_Other
     11               ohe__education_level_PhD
     12        ohe__employment_status_Employed
     13         ohe__employment_status_Retired
     14   ohe__employment_status_Self-employed
     15         ohe__employment_status_Student
     16      ohe__employment_status_Unemployed
     17             ohe__loan_purpose_Business
     18                  ohe__loan_purpose_Car
     19   ohe__loan_purpose_Debt consolidation
     20            ohe__loan_purpose_Education
     21                 ohe__loan_purpose_Home
     22              ohe__loan_purpose_Medical
     23                ohe__loan_purpose_Other
     24             ohe__loan_purpose_Vacation
pca  0                               pca__pca0
     1                               pca__pca1
     2                               pca__pca2
     3                               pca__pca3
     4                               pca__pca4

,name
0,lr3__loan_paid_back_0
1,lr3__loan_paid_back_1


In [55]:
Markdown(
    e.desc_node('cb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_cb1["cb1"]
        cb1_dummy[ ]
        style cb1_dummy fill:none,stroke:none
    end
    style node_cb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_cb1
```

**Path from Root to 'cb1' (1 path(s) found)**

In [56]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_xgb["xgb"]
            grp_xgb_count["1 node(s)"]
            style grp_xgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_xgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [57]:
Markdown(
    e.desc_node('lgb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lgb1["lgb1"]
        lgb1_dummy[ ]
        style lgb1_dummy fill:none,stroke:none
    end
    style node_lgb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_lgb1
```

**Path from Root to 'lgb1' (1 path(s) found)**

In [58]:
e._find_descendants('std')

{'lr1', 'lr2', 'lr3', 'pca'}

In [59]:
# e = Experimenter(df_train, sp = skf, sp_v = None, splitter_params = {'y': y})

In [60]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_xgb["xgb"]
            grp_xgb_count["1 node(s)"]
            style grp_xgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_xgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [61]:
from sklearn.preprocessing import TargetEncoder

e.set_node('tgt', 'preprocessor', TargetEncoder, edges = [(None, X_cat), (None, y)], y = y, params={'target_type': 'binary'}, method = 'fit_transform')

In [62]:
e.build()

Building 1 node(s)
Build 3/3 (100%)> tgt 1/1 (100%)
Build complete: 1 node(s)


In [63]:
e.set_node('lr4', 'lr', LogisticRegression, edges =  [('std', None), ('ohe', None), ('tgt', None), ('pca', None)])

In [64]:
e.exp(None)

Experimenting 1 node(s)
Exp 3/3 (100%)> lr4 1/1 (100%)
Experimentation complete: 1 node(s)


In [65]:
Markdown(
    e.desc_node('lr4', direction = 'LR', show_params=True)
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr4["lr4"]
        lr4_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr>"]
    end
    style node_lr4 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>PCA</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>n_components</b></td><td align='left'>0.9</td></tr></table>"]
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_tgt["tgt"]
        tgt_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>TargetEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>fit_transform</td></tr><tr><td align='left'><b>target_type</b></td><td align='left'>binary</td></tr></table>"]
    end
    style node_tgt fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr4
    Root --> node_ohe
    Root --> node_std
    Root --> node_tgt
    node_ohe --> node_lr4
    node_pca --> node_lr4
    node_std --> node_lr4
    node_std --> node_pca
    node_tgt --> node_lr4
```

**Path from Root to 'lr4' (5 path(s) found)**

In [67]:
e.desc_node_vars('lr4', 0)

(                                          name
 node seq                                      
 std  0                      std__annual_income
      1               std__debt_to_income_ratio
      2                       std__credit_score
      3                        std__loan_amount
      4                      std__interest_rate
      5                  std__grade_subgrade_no
 ohe  0                      ohe__gender_Female
      1                        ohe__gender_Male
      2                       ohe__gender_Other
      3            ohe__marital_status_Divorced
      4             ohe__marital_status_Married
      5              ohe__marital_status_Single
      6             ohe__marital_status_Widowed
      7         ohe__education_level_Bachelor's
      8        ohe__education_level_High School
      9           ohe__education_level_Master's
      10             ohe__education_level_Other
      11               ohe__education_level_PhD
      12        ohe__employment_status_E

In [68]:
e.metric['AUC'].get_metrics(None).stack(level=1).groupby(level=0).mean().stack(level=0).groupby(level=0).mean()

,valid,train_sub,valid_sub
cb1,0.916022,0.948672,0.921615
lgb1,0.902589,0.999661,0.902714
lr1,0.777864,0.779638,0.783745
lr2,0.916235,0.920090,0.916969
lr3,0.913964,0.918438,0.917568
lr4,0.915792,0.919979,0.915958
xgb1,0.893260,0.999999,0.885357


In [69]:
l = list()
for no, i in enumerate(e.get_results('lr4', 'coef')):
    l.append(pd.concat([j.rename(no_i) for no_i, j in enumerate(i)], axis = 1).stack().rename(no))
pd.concat(l, axis=1)

0         1         2
0 std__annual_income                   0 -0.033025  0.010999  0.015944
  std__debt_to_income_ratio            0 -0.411915 -0.438788 -0.460908
  std__credit_score                    0  0.823460  0.930676  0.929247
  std__loan_amount                     0 -0.024339 -0.001498 -0.005384
  std__interest_rate                   0  0.013250 -0.005225  0.057839
  std__grade_subgrade_no               0  0.412014  0.545567  0.480441
  ohe__gender_Female                   0 -0.042005  0.051464 -0.270856
  ohe__gender_Male                     0  0.020618  0.114465 -0.188129
  ohe__gender_Other                    0  0.079771 -0.075006  0.623725
  ohe__marital_status_Divorced         0 -0.088478  0.102976  0.386799
  ohe__marital_status_Married          0  0.006712 -0.305734 -0.041850
  ohe__marital_status_Single           0 -0.026422 -0.298124 -0.058925
  ohe__marital_status_Widowed          0  0.166572  0.591804 -0.121284
  ohe__education_level_Bachelor's      0 -0.164071 -0.096048 -0.311216
  ohe__education_level_High School     0 -0.078361 -0.185629 -0.151342
  ohe__education_level_Master's        0 -0.249998 -0.200558 -0.272649
  ohe__education_level_Other           0 -0.180888  0.080403  0.132451
  ohe__education_level_PhD             0  0.731702  0.492754  0.767498
  ohe__employment_status_Employed      0  0.643883  0.626176  0.879320
  ohe__employment_status_Retired       0  2.050646  2.050790  2.016697
  ohe__employment_status_Self-employed 0  0.604500  0.677723  0.483096
  ohe__employment_status_Student       0 -1.146179 -0.966311 -1.049858
  ohe__employment_status_Unemployed    0 -2.094466 -2.297455 -2.164515
  ohe__loan_purpose_Business           0 -0.305911 -0.132935 -0.506783
  ohe__loan_purpose_Car                0  0.499547  0.105626  0.431023
  ohe__loan_purpose_Debt consolidation 0 -0.258538 -0.217337 -0.239093
  ohe__loan_purpose_Education          0 -0.000239  0.096673 -0.112516
  ohe__loan_purpose_Home               0  0.045503  0.257998  0.059202
  ohe__loan_purpose_Medical            0 -0.168027 -0.098580  0.165469
  ohe__loan_purpose_Other              0 -0.032139 -0.200756 -0.292288
  ohe__loan_purpose_Vacation           0  0.278187  0.280234  0.659727
  tgt__gender                          0 -0.033241 -0.003969  0.137270
  tgt__marital_status                  0 -0.062529  0.079510 -0.041869
  tgt__education_level                 0  0.012686 -0.016570  0.121039
  tgt__employment_status               0  2.655379  2.632965  2.477929
  tgt__loan_purpose                    0 -0.137723 -0.056563 -0.198710
  pca__pca0                            0  0.280733  0.278772  0.288673
  pca__pca1                            0 -0.249690  0.137216 -0.178131
  pca__pca2                            0 -0.205200  0.066333 -0.282634
  pca__pca3                            0 -0.244354 -0.395430 -0.303433
  pca__pca4                            0  0.096828  0.066138  0.136759
  intercept                            0  0.057852  0.102740  0.244009

In [70]:
e.get_results_merge('cb1', 'feature_importances_pvc', agg_outer = True).sort_values(ascending = False)

employment_status       24.580928
debt_to_income_ratio    23.124728
credit_score            15.595137
grade_subgrade_no        6.537125
loan_amount              5.431739
interest_rate            5.167439
annual_income            5.007924
loan_purpose             4.294083
education_level          3.546903
gender                   3.517049
marital_status           3.196945
dtype: float64

In [71]:
e.get_results_merge('cb1', 'feature_importances_interaction', agg_outer = True).unstack().fillna(0).pipe(
    lambda x: x.iloc[np.argsort(-x.sum(axis=1)), np.argsort(-x.sum(), axis=0)]
)

feat2,employment_status,loan_purpose,education_level,marital_status,interest_rate,gender,grade_subgrade_no,credit_score,loan_amount,debt_to_income_ratio
feat1,,,,,,,,,,
debt_to_income_ratio,5.639454,1.818698,1.800063,2.281955,3.112141,2.115557,3.026139,4.619111,2.405804,0.000000
credit_score,4.832118,1.738930,1.743565,1.652774,2.010338,1.588778,0.962546,0.000000,1.901645,0.000000
annual_income,2.173131,0.867756,0.749699,0.880445,0.928675,0.831484,0.652076,1.821215,1.084547,2.590267
loan_amount,1.913514,0.827809,1.194900,0.998190,1.556591,1.095899,0.898544,0.000000,0.000000,0.000000
interest_rate,1.877057,1.375989,1.216270,1.072896,0.000000,1.017758,1.136219,0.000000,0.000000,0.000000
gender,2.573196,1.511113,1.465362,1.240001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
grade_subgrade_no,3.026792,0.819812,0.887864,0.849679,0.000000,0.821621,0.000000,0.000000,0.000000,0.000000
marital_status,2.551013,1.258556,1.254627,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
education_level,2.021308,1.338511,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [72]:
import seaborn as sns
e.get_results_merge('cb1', 'evals_result', agg_outer = True).unstack(level=[-2, -1])

Logloss                          
        learn validation_0 validation_1
0    0.645487     0.645316     0.644188
1    0.613548     0.613195     0.611324
2    0.572388     0.572043     0.569495
3    0.538093     0.537632     0.534533
4    0.513244     0.512752     0.509031
..        ...          ...          ...
995  0.091122     0.119302     0.254643
996  0.091039     0.119214     0.254645
997  0.090959     0.119141     0.254650
998  0.090890     0.119096     0.254755
999  0.090778     0.119019     0.254920

[1000 rows x 3 columns]

In [73]:
e.get_results_merge('xgb1', 'evals_result', agg_outer = True).unstack(level=[-2, -1])

logloss             
   validation_0 validation_1
0      0.374802     0.384127
1      0.318155     0.332784
2      0.282461     0.303267
3      0.257181     0.284705
4      0.238229     0.272196
..          ...          ...
95     0.042537     0.325361
96     0.041968     0.325900
97     0.041214     0.325812
98     0.040574     0.325562
99     0.040124     0.325673

[100 rows x 2 columns]

In [74]:
e.get_results_merge('lgb1', 'evals_result', agg_outer = True).unstack(level=[-2, -1])

binary_logloss          
         training   valid_1
0        0.445450  0.447038
1        0.408066  0.411590
2        0.379432  0.384363
3        0.356291  0.363595
4        0.337015  0.345947
..            ...       ...
95       0.080419  0.271584
96       0.079727  0.272573
97       0.078764  0.272998
98       0.077914  0.273085
99       0.076997  0.273688

[100 rows x 2 columns]

In [75]:
from modeler import create_like

In [76]:
e2 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = ss, sp_v = StratifiedKFold(2, random_state=123, shuffle = True),
                splitter_params = {'y': y})

TypeError: create_like() missing 1 required positional argument: 'path'

In [76]:
e2.build()

NameError: name 'e2' is not defined

In [77]:
lr_a.get_coef('lr1').T.groupby(level = [2]).mean().T

NameError: name 'lr_a' is not defined

In [ ]:
lr_a.get_intercept('lr1').T.groupby(level = [2]).mean().T

In [ ]:
e.root.data

In [ ]:
for i in e2.get_node_valid_output(0, 'cb1', slice(0, -1)):
    print(i.data)

In [ ]:
e2.get_data_valid(0, [('lr1', slice(0, -1))])

In [ ]:
for i in e2.get_data_valid(0, [('lr1', slice(0, -1))]):
    print(i.data)

In [ ]:
e3 = create_like(e, df_train.sample(frac = 0.1, random_state = 123), splitter_params = {'y': y})

In [ ]:
e3.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
e3.set_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')

In [ ]:
e3.set_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', Noneset_grp

In [ ]:
e3.set_node('lr1', 'lr')

In [ ]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    #sgpp.PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [ ]:
X_all = df_test.columns
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [ ]:
e = Experimenter(df_train, sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [ ]:
e.set_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')
e.set_grp('preprocessor', method = 'transform')

In [ ]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
for a, b in e.get_data(0, [(None, X_num)]):
    print(a[0].data, b)

In [ ]:
from sklearn.linear_model import LogisticRegression

e.set_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e.set_node('lr1', 'lr')

In [ ]:
for i in e.get_data_valid(0, [(None, y)]):
    print(i.data)

In [ ]:
ss = StratifiedKFold(n_splits=3)
for a, b in ss.split(df_train[X_all],  df_train[y]):
    pass

In [ ]:
lr = LogisticRegression()
lr.fit(df_train[X_num], df_train[[y]])

In [ ]:
df_train.to_pandas()[[y]].shape

In [ ]:
e.nodes['lr1'].y